# Feature Selection for the atmospheric inputs

In [ ]:
import logging
import keras_tuner
import keras
import tensorflow as tf
import time
import pathlib
import os



from usl_models.atmo_ml.model import AtmoModel
from usl_models.atmo_ml import dataset, visualizer, vars

for gpu in tf.config.list_physical_devices('GPU'):
    tf.config.experimental.set_memory_growth(gpu, True)

logging.getLogger().setLevel(logging.WARNING)
keras.utils.set_random_seed(812)
visualizer.init_plt()


In [ ]:
import keras
from keras import layers
import tensorflow as tf
import numpy as np
import pandas as pd

class MaskedResidualT2SpatialRefiner(keras.Model):
    def __init__(self, st_mask, spatial_mask, use_lu=True):
        super().__init__()

        # save masks as tensors
        self.st_mask = tf.constant(st_mask, dtype=tf.float32)          # [12]
        self.spatial_mask = tf.constant(spatial_mask, dtype=tf.float32)  # [22]
        self.use_lu = use_lu

        # main tower
        self.conv1 = layers.Conv2D(64, 3, padding="same", activation="relu")
        self.conv2 = layers.Conv2D(64, 3, padding="same", activation="relu")
        self.conv3 = layers.Conv2D(32, 3, padding="same", activation="relu")
        self.conv4 = layers.Conv2D(32, 3, padding="same", activation="relu")

        # detail branch
        self.detail_conv1 = layers.Conv2D(32, 3, padding="same", activation="relu")
        self.detail_conv2 = layers.Conv2D(32, 3, padding="same", activation="relu")

        # output heads
        self.residual_out_t0 = layers.Conv2D(1, 1, padding="same", activation="linear")
        self.residual_out_t1 = layers.Conv2D(1, 1, padding="same", activation="linear")

        self.raw_scale_t0 = self.add_weight(
            name="raw_scale_t0", shape=(), initializer="zeros", trainable=True
        )
        self.raw_scale_t1 = self.add_weight(
            name="raw_scale_t1", shape=(), initializer="zeros", trainable=True
        )

        self.avg_pool = layers.AveragePooling2D(pool_size=5, strides=1, padding="same")

    def _normalize_hw(self, x):
        mean = tf.reduce_mean(x, axis=[1, 2], keepdims=True)
        std = tf.math.reduce_std(x, axis=[1, 2], keepdims=True) + 1e-6
        return (x - mean) / std

    def _high_pass(self, x):
        smooth = self.avg_pool(x)
        return x - smooth

    def call(self, inputs):
        tt_idx = vars.Spatiotemporal.TT.value

        # -----------------------------
        # Baseline always uses original TT
        # -----------------------------
        tt = inputs["spatiotemporal"][:, :, :, :, tt_idx]  # [B,T,H,W]
        base_t0 = tt[:, -2, :, :][:, tf.newaxis, :, :, tf.newaxis]
        base_t1 = tt[:, -1, :, :][:, tf.newaxis, :, :, tf.newaxis]

        # -----------------------------
        # Masked refinement inputs
        # -----------------------------
        st = inputs["spatiotemporal"]  # [B,T,H,W,12]
        st = st * self.st_mask[tf.newaxis, tf.newaxis, tf.newaxis, tf.newaxis, :]
        st = tf.transpose(st, [0, 2, 3, 1, 4])  # [B,H,W,T,C]
        st = tf.reshape(st, [tf.shape(st)[0], tf.shape(st)[1], tf.shape(st)[2], -1])
        st = self._normalize_hw(st)

        spatial = inputs["spatial"]  # [B,H,W,22]
        spatial = spatial * self.spatial_mask[tf.newaxis, tf.newaxis, tf.newaxis, :]
        spatial = self._normalize_hw(spatial)

        if self.use_lu:
            lu = tf.cast(inputs["lu_index"], tf.float32)[..., tf.newaxis]
            lu = self._normalize_hw(lu)
        else:
            lu = tf.zeros_like(tf.cast(inputs["lu_index"], tf.float32))[..., tf.newaxis]

        fused = tf.concat([st, spatial, lu], axis=-1)

        # main tower
        x = self.conv1(fused)
        x = self.conv2(x)
        x = self.conv3(x)
        x = self.conv4(x)

        # high-pass detail branch
        detail_in = tf.concat([
            self._high_pass(st),
            self._high_pass(spatial),
            self._high_pass(lu)
        ], axis=-1)

        d = self.detail_conv1(detail_in)
        d = self.detail_conv2(d)

        x = tf.concat([x, d], axis=-1)

        # residuals
        res_t0 = self.residual_out_t0(x)[:, tf.newaxis, :, :, :]
        res_t1 = self.residual_out_t1(x)[:, tf.newaxis, :, :, :]

        # small bounded scales
        scale_t0 = 0.01 * tf.sigmoid(self.raw_scale_t0)
        scale_t1 = 0.01 * tf.sigmoid(self.raw_scale_t1)

        pred_t0 = base_t0 + scale_t0 * res_t0
        pred_t1 = base_t1 + scale_t1 * res_t1

        return tf.concat([pred_t0, pred_t1], axis=1)

In [ ]:
def t2_loss():
    return keras.losses.MeanAbsoluteError()

def compute_dataset_metrics(model, ds_eval):
    preds = []
    labels = []

    for x_batch, y_batch in ds_eval:
        p = model.predict(x_batch, verbose=0)
        preds.append(p)
        labels.append(y_batch.numpy())

    preds = np.concatenate(preds, axis=0)
    labels = np.concatenate(labels, axis=0)

    mse = np.mean((preds - labels) ** 2)
    mae = np.mean(np.abs(preds - labels))
    return {"mse": float(mse), "mae": float(mae)}

In [ ]:
ST_FEATURES = [v.name for v in vars.Spatiotemporal]   # 12 names
SPATIAL_FEATURES = [f"spatial_{i}" for i in range(22)]
LU_FEATURE = ["lu_index"]

ALL_FEATURES = (
    [("st", i, ST_FEATURES[i]) for i in range(len(ST_FEATURES))] +
    [("spatial", i, SPATIAL_FEATURES[i]) for i in range(len(SPATIAL_FEATURES))] +
    [("lu", 0, "lu_index")]
)

print("Spatiotemporal:", ST_FEATURES)
print("Spatial:", SPATIAL_FEATURES)
print("LU:", LU_FEATURE)

In [ ]:
def build_feature_masks(selected_features):
    st_mask = np.zeros(len(ST_FEATURES), dtype=np.float32)
    spatial_mask = np.zeros(len(SPATIAL_FEATURES), dtype=np.float32)
    use_lu = False

    for kind, idx, name in selected_features:
        if kind == "st":
            st_mask[idx] = 1.0
        elif kind == "spatial":
            spatial_mask[idx] = 1.0
        elif kind == "lu":
            use_lu = True

    return st_mask, spatial_mask, use_lu

In [ ]:
def run_feature_subset(
    selected_features,
    train_ds,
    val_ds_fit,
    val_ds_eval,
    epochs=10,
    steps_per_epoch=3,
    validation_steps=1,
    learning_rate=1e-3,
    verbose=0,
):
    st_mask, spatial_mask, use_lu = build_feature_masks(selected_features)

    model = MaskedResidualT2SpatialRefiner(
        st_mask=st_mask,
        spatial_mask=spatial_mask,
        use_lu=use_lu,
    )

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate),
        loss=t2_loss(),
        metrics=[
            keras.metrics.MeanAbsoluteError(),
            keras.metrics.RootMeanSquaredError(),
        ],
        run_eagerly=True,   # stable for notebook/debug
    )

    history = model.fit(
        train_ds,
        validation_data=val_ds_fit,
        epochs=epochs,
        steps_per_epoch=steps_per_epoch,
        validation_steps=validation_steps,
        verbose=verbose,
    )

    metrics = compute_dataset_metrics(model, val_ds_eval)

    learned_scale_t0 = float((0.01 * tf.sigmoid(model.raw_scale_t0)).numpy())
    learned_scale_t1 = float((0.01 * tf.sigmoid(model.raw_scale_t1)).numpy())

    return {
        "model": model,
        "history": history,
        "selected_features": [name for _, _, name in selected_features],
        "metrics": metrics,
        "scale_t0": learned_scale_t0,
        "scale_t1": learned_scale_t1,
    }

In [ ]:
def greedy_forward_feature_selection(
    train_ds,
    val_ds_fit,
    val_ds_eval,
    candidate_features=ALL_FEATURES,
    max_features=10,
    min_improvement=1e-5,
    epochs=10,
    steps_per_epoch=3,
    validation_steps=1,
):
    selected = []
    remaining = candidate_features.copy()
    search_log = []

    # baseline: no refinement inputs
    baseline_result = run_feature_subset(
        selected_features=selected,
        train_ds=train_ds,
        val_ds_fit=val_ds_fit,
        val_ds_eval=val_ds_eval,
        epochs=epochs,
        steps_per_epoch=steps_per_epoch,
        validation_steps=validation_steps,
        verbose=0,
    )
    best_score = baseline_result["metrics"]["mse"]
    best_result = baseline_result

    search_log.append({
        "step": 0,
        "added_feature": None,
        "selected_features": [],
        "val_mse": best_score,
        "val_mae": baseline_result["metrics"]["mae"],
        "scale_t0": baseline_result["scale_t0"],
        "scale_t1": baseline_result["scale_t1"],
    })

    print(f"[step 0] baseline mse={best_score:.8f} mae={baseline_result['metrics']['mae']:.8f}")

    for step in range(1, max_features + 1):
        trial_results = []

        for feat in remaining:
            trial_selected = selected + [feat]

            result = run_feature_subset(
                selected_features=trial_selected,
                train_ds=train_ds,
                val_ds_fit=val_ds_fit,
                val_ds_eval=val_ds_eval,
                epochs=epochs,
                steps_per_epoch=steps_per_epoch,
                validation_steps=validation_steps,
                verbose=0,
            )

            trial_results.append((feat, result))

        # choose best feature addition
        feat_best, result_best = min(trial_results, key=lambda x: x[1]["metrics"]["mse"])
        new_score = result_best["metrics"]["mse"]
        improvement = best_score - new_score

        print(
            f"[step {step}] best add={feat_best[2]} "
            f"mse={new_score:.8f} mae={result_best['metrics']['mae']:.8f} "
            f"improvement={improvement:.8f}"
        )

        if improvement < min_improvement:
            print("Stopping: no meaningful improvement.")
            break

        selected.append(feat_best)
        remaining.remove(feat_best)

        best_score = new_score
        best_result = result_best

        search_log.append({
            "step": step,
            "added_feature": feat_best[2],
            "selected_features": [name for _, _, name in selected],
            "val_mse": result_best["metrics"]["mse"],
            "val_mae": result_best["metrics"]["mae"],
            "scale_t0": result_best["scale_t0"],
            "scale_t1": result_best["scale_t1"],
        })

    log_df = pd.DataFrame(search_log)
    return best_result, log_df

In [ ]:
filecache_dir = pathlib.Path("/home/shared/climateiq/filecache")
small_example_keys = [
    ("NYC_Heat_Test/NYC_summer_2000_01p", "2000-05-25"),
    ("NYC_Heat_Test/NYC_summer_2000_01p", "2000-05-26"),
    ("NYC_Heat_Test/NYC_summer_2000_01p", "2000-05-27"),
    ("NYC_Heat_Test/NYC_summer_2000_01p", "2000-05-28"),
    ("PHX_Heat_Test/PHX_summer_2008_25p", "2008-05-25"),
    ("PHX_Heat_Test/PHX_summer_2008_25p", "2008-05-26"),
    ("PHX_Heat_Test/PHX_summer_2008_25p", "2008-05-27"),
    ("PHX_Heat_Test/PHX_summer_2008_25p", "2008-05-28"),
]

train_keys = small_example_keys[:6]
val_keys = small_example_keys[6:]
ds_config_t2 = dataset.Config(
    output_timesteps=2,
    sto_vars=(vars.SpatiotemporalOutput.T2,),
)

# fit datasets (small proxy)
train_ds_t2_search = dataset.load_dataset_cached(
    filecache_dir=filecache_dir,
    example_keys=train_keys,
    config=ds_config_t2,
    shuffle=True,
).batch(2).repeat()

val_ds_t2_fit = dataset.load_dataset_cached(
    filecache_dir=filecache_dir,
    example_keys=val_keys,
    config=ds_config_t2,
    shuffle=False,
).batch(2).repeat()

# real evaluation dataset (no repeat)
val_ds_t2_eval = dataset.load_dataset_cached(
    filecache_dir=filecache_dir,
    example_keys=val_keys,
    config=ds_config_t2,
    shuffle=False,
).batch(1)

In [ ]:
best_result, search_log_df = greedy_forward_feature_selection(
    train_ds=train_ds_t2_search,
    val_ds_fit=val_ds_t2_fit,
    val_ds_eval=val_ds_t2_eval,
    max_features=8,          # start small
    min_improvement=1e-5,
    epochs=10,               # proxy search budget
    steps_per_epoch=3,
    validation_steps=1,
)

search_log_df

In [ ]:
print("Best selected features:", best_result["selected_features"])
print("Best val MSE:", best_result["metrics"]["mse"])
print("Best val MAE:", best_result["metrics"]["mae"])
print("Learned scales:", best_result["scale_t0"], best_result["scale_t1"])

In [ ]:
winner_feature_names = best_result["selected_features"]
print("Winner feature names:", winner_feature_names)

# rebuild selected tuple list
winner_selected = []
for feat in ALL_FEATURES:
    if feat[2] in winner_feature_names:
        winner_selected.append(feat)

final_result = run_feature_subset(
    selected_features=winner_selected,
    train_ds=train_ds_t2_search,
    val_ds_fit=val_ds_t2_fit,
    val_ds_eval=val_ds_t2_eval,
    epochs=30,
    steps_per_epoch=3,
    validation_steps=1,
    verbose=1,
)

print("Final best subset:", final_result["selected_features"])
print("Final val MSE:", final_result["metrics"]["mse"])
print("Final val MAE:", final_result["metrics"]["mae"])
print("Final scales:", final_result["scale_t0"], final_result["scale_t1"])

In [ ]:
best_model = final_result["model"]

all_preds = []
all_labels = []
all_inputs = []

for x_batch, y_batch in val_ds_t2_eval:
    pred_batch = best_model.predict(x_batch, verbose=0)
    all_preds.append(pred_batch)
    all_labels.append(y_batch.numpy())
    all_inputs.append(x_batch)

all_preds = np.concatenate(all_preds, axis=0)
all_labels = np.concatenate(all_labels, axis=0)

In [ ]:
tt_idx = vars.Spatiotemporal.TT.value
all_baselines = []

for x_batch, _ in val_ds_t2_eval:
    tt = x_batch["spatiotemporal"][:, :, :, :, tt_idx]
    base_t0 = tt[:, -2, :, :]
    base_t1 = tt[:, -1, :, :]
    baseline = tf.stack([base_t0, base_t1], axis=1)[..., tf.newaxis]
    all_baselines.append(baseline.numpy())

all_baselines = np.concatenate(all_baselines, axis=0)

In [ ]:
import matplotlib.pyplot as plt

def plot_triplet(labels, baselines, preds, example_idx=0, timestep=0):
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    axes[0].imshow(labels[example_idx, timestep, :, :, 0], origin="lower")
    axes[0].set_title(f"label ex={example_idx} t={timestep}")

    axes[1].imshow(baselines[example_idx, timestep, :, :, 0], origin="lower")
    axes[1].set_title(f"persistence ex={example_idx} t={timestep}")

    axes[2].imshow(preds[example_idx, timestep, :, :, 0], origin="lower")
    axes[2].set_title(f"best subset pred ex={example_idx} t={timestep}")

    plt.tight_layout()
    plt.show()

In [ ]:
plot_triplet(all_labels, all_baselines, all_preds, example_idx=0, timestep=0)
plot_triplet(all_labels, all_baselines, all_preds, example_idx=0, timestep=1)